# Phase-2 Step 8: Explainability & Diagnostics

**Objective:** Generate interpretable explanations for predictions.

**Methods:**
1. **Feature Importance:** From tree-based models
2. **SHAP Values:** SHapley Additive exPlanations
3. **Partial Dependence:** Feature-prediction relationships
4. **Sample Explanations:** Per-prediction diagnostics

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os
import warnings
warnings.filterwarnings('ignore')



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay

from data_splits import create_within_battery_split, normalize_features

print("✅ Setup complete")

In [ ]:
# Check for SHAP
!pip install shap
try:
    import shap
    HAVE_SHAP = True
    print("✅ SHAP available")
except ImportError:
    HAVE_SHAP = False
    print("⚠️ SHAP not installed. Install with: pip install shap")

## 1. Load Data and Train Model

In [ ]:
# Load data
all_features = pd.read_csv('data/phase2/harmonized/all_features.csv')

# Split
splits = create_within_battery_split(all_features, train_ratio=0.7, val_ratio=0.15)
train_df, val_df, test_df = splits['train'], splits['val'], splits['test']

# Features
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
feature_cols = [c for c in all_features.columns if c not in exclude_cols]

# Normalize
train_norm, val_norm, test_norm, _ = normalize_features(train_df, val_df, test_df, exclude_cols)

# Prepare data
X_train = np.vstack([train_norm[feature_cols].values, val_norm[feature_cols].values])
y_train = np.hstack([train_norm['soh'].values, val_norm['soh'].values])
X_test = test_norm[feature_cols].values
y_test = test_norm['soh'].values

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {len(feature_cols)}")

In [ ]:
# Train Gradient Boosting (for explainability)
model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2)

print(f"Model R²: {r2:.4f}")

## 2. Feature Importance (Built-in)

In [ ]:
# Get feature importance
importance = model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

print("\nTop 10 Features by Importance:")
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(10, 8))

top_n = 15
top_features = importance_df.head(top_n)

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, top_n))[::-1]

bars = ax.barh(range(top_n), top_features['importance'], color=colors, edgecolor='black')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature Importance (Gradient Boosting)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add values
for bar, val in zip(bars, top_features['importance']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/phase2/figures/step8_feature_importance.png', dpi=150)
plt.show()

print("✅ Saved: results/phase2/figures/step8_feature_importance.png")

## 3. SHAP Analysis

In [ ]:
if HAVE_SHAP:
    print("Computing SHAP values...")
    
    # Use a subset for speed
    X_sample = X_test[:50] if len(X_test) > 50 else X_test
    
    # Create explainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    print(f"SHAP values shape: {shap_values.shape}")
else:
    print("SHAP not available - skipping")

In [ ]:
if HAVE_SHAP:
    # Summary plot
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
    plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/phase2/figures/step8_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: results/phase2/figures/step8_shap_summary.png")

In [ ]:
if HAVE_SHAP:
    # Bar plot
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (Mean |SHAP|)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/phase2/figures/step8_shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: results/phase2/figures/step8_shap_bar.png")

## 4. Partial Dependence Plots

In [ ]:
# Get top 4 features for PDP
top_4_features = importance_df.head(4)['feature'].tolist()
top_4_indices = [feature_cols.index(f) for f in top_4_features]

print(f"Generating PDPs for: {top_4_features}")

In [ ]:
# Partial dependence plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (feature_idx, feature_name) in enumerate(zip(top_4_indices, top_4_features)):
    ax = axes.flat[idx]
    
    PartialDependenceDisplay.from_estimator(
        model, X_train, [feature_idx],
        feature_names=feature_cols,
        ax=ax
    )
    ax.set_title(f'Partial Dependence: {feature_name}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Partial Dependence Plots (Top 4 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/step8_partial_dependence.png', dpi=150)
plt.show()

print("✅ Saved: results/phase2/figures/step8_partial_dependence.png")

## 5. Sample Explanations

In [ ]:
# Generate text explanations for sample predictions
from feature_to_text import FeatureToText

converter = FeatureToText()

# Select samples: best, worst, median predictions
errors = np.abs(y_test - y_pred)
best_idx = np.argmin(errors)
worst_idx = np.argmax(errors)
median_idx = np.argsort(errors)[len(errors)//2]

sample_indices = [best_idx, median_idx, worst_idx]
sample_labels = ['Best Prediction', 'Median Prediction', 'Worst Prediction']

print("\n" + "="*70)
print("SAMPLE PREDICTION EXPLANATIONS")
print("="*70)

for idx, label in zip(sample_indices, sample_labels):
    print(f"\n--- {label} ---")
    print(f"Actual SOH: {y_test[idx]:.2f}%")
    print(f"Predicted SOH: {y_pred[idx]:.2f}%")
    print(f"Error: {errors[idx]:.2f}%")
    
    # Get feature values
    row = test_df.iloc[idx]
    desc = converter.convert_single(row.to_dict())
    
    print(f"\nDiagnosis: {desc.diagnosis}")
    print(f"\nFull description:")
    print(desc.full_description)

## 6. Save Results

In [ ]:
# Save feature importance
importance_df.to_csv('results/phase2/step8_feature_importance.csv', index=False)
print("✅ Saved: results/phase2/step8_feature_importance.csv")

In [ ]:
print("\n" + "="*70)
print("STEP 8: EXPLAINABILITY SUMMARY")
print("="*70)

print("\n📊 Key Insights:")
print(f"\n1. Top 5 Most Important Features:")
for i, row in importance_df.head(5).iterrows():
    print(f"   {row['feature']}: {row['importance']:.4f}")

print("\n2. Feature Categories by Importance:")
# Group by category
categories = {
    'Capacity/Physics': ['capacity', 'discharge_time', 'data_quality'],
    'Voltage': ['voltage_mean', 'voltage_std', 'voltage_min', 'voltage_max', 'voltage_range'],
    'ICA': ['ica_peak_voltage', 'ica_peak_magnitude', 'ica_mean', 'ica_std'],
    'Temperature': ['temp_mean', 'temp_max', 'temp_delta', 'temp_std'],
    'Current': ['current_mean', 'current_std']
}

for cat_name, cat_features in categories.items():
    cat_importance = importance_df[importance_df['feature'].isin(cat_features)]['importance'].sum()
    print(f"   {cat_name}: {cat_importance:.4f}")

print("\n✅ Step 8 Complete - Proceed to Step 9: Statistical Checks")